### 1. Import the dataset as well as all required libraries

In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np, random
from torchvision.datasets import FashionMNIST
from torchvision import transforms
from torch.utils.data import DataLoader, random_split, Subset
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

### 2. Set the random seed values to 123
So that you are comparing the same set of data with other students

In [24]:
torch.manual_seed(123)
np.random.seed(123)
random.seed(123)

In [25]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

### 3. Load the subset of MNIST Fashion dataset using PyTorch tensors

In [26]:
#FashionMNIST = FashionMNIST(root='.data', train=True, download=True, transform=transform)
train_full = FashionMNIST(root='.data', train=True, download=True, transform=transform)
test_full = FashionMNIST(root='.data', train=True, download=True, transform=transform)

### 4. Split the subset of the MNIST Fashion dataset into training and test set using PyTorch DataLoader

In [27]:
train_subset = Subset(train_full, range(10000))
test_subset = Subset(test_full, range(2000))
train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=False)

### 5. Implement a Multilayer Perceptron (MLP) using PyTorch to create a classification model for the fashion items in the MNIST Fashion dataset

In [28]:
class MLP(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)   # Flatten images
        return self.net(x)

### 6. Choose one hyperparameter to tune and explain why the chosen hyperparameter

In [29]:
# Hyperparameter tuned. Hidden layer size (it is chosen because it directly controls model capacity)
hidden_size=256
model=MLP(hidden_size)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

for epoch in range(10):
    model.train()
    for x, y in train_loader:
        optimizer.zero_grad()
        loss=criterion(model(x),y)
        loss.backward()
        optimizer.step()

def evaluate_model(model,test_loader):
    model.eval()
    all_preds=[]
    all_labels=[]
    with torch.no_grad():
        for inputs,labels in test_loader:
            outputs=model(inputs)
            _,preds=torch.max(outputs,1)
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())
    return all_preds, all_labels


### 7. Calculate and report the accuracy, precision, recall and F1-score using the 'macro' average in the appropriate functions from sklearn

In [30]:
preds, labels=evaluate_model(model,test_loader)
cm=confusion_matrix(labels, preds)
accuracy = accuracy_score(labels, preds)
precision = precision_score(labels, preds, average="macro")
recall = recall_score(labels, preds, average="macro")
f1 = f1_score(labels, preds, average="macro")
print("Accuracy", accuracy)
print("Precision", precision)
print("Recall", recall)
print("F1",f1)
print(cm)

Accuracy 0.921
Precision 0.920651652253091
Recall 0.9197044794256332
F1 0.9191124407549349
[[183   0   2   3   0   0   6   0   0   0]
 [  3 211   0   2   0   0   0   0   0   0]
 [  2   0 175   3  10   1  10   1   0   0]
 [  4   0   0 183   6   0   2   0   0   0]
 [  1   0  12   2 163   1   7   0   0   0]
 [  0   0   0   0   0 188   0   9   0   3]
 [ 29   0  10   3   6   0 144   0   1   1]
 [  0   0   0   0   0   0   0 204   0  11]
 [  0   0   0   5   0   0   0   0 191   2]
 [  0   0   0   0   0   0   0   0   0 200]]
